## Introduction

This notebook presents a partial, modular workflow for analyzing air quality data (CO₂) using the classes and tools implemented in the **Sourcers** package. The goal is to show, in a clear and reproducible way, how these functions are used within a real analysis pipeline.

The following steps are illustrated throughout the notebook:

- **Loading data** from a remote source using `DataLoad`.
- **Automatically cleaning** null values and inconsistencies using `DataCleaner`.
- **Building time series** from lists of readings and timestamps with `TimeSeries`.
- **Extracting statistical and temporal features** using `FeatureExtractor` and `TemporalFeatureExtractor`.
- **Classifying observations** into different categories: CO₂ levels, ventilation type, and time of day.
- **Visualizing data** using histograms, boxplots, violins, and bar charts with `PlotBuilder`.
- **Statistical evaluation of groups** using tests such as Kruskal–Wallis.
- **Exporting results** in `.csv` and `.png` formats using `TableExporter` and `FigureExporter`.

This document serves as a practical guide to understanding how each component works and how to integrate them into a structured and scalable analysis.

_____________________________________________________________________________________________________________________

### 1. Importing libraries and modules

In this section, standard libraries (such as os, time, pandas) are loaded and the modules developed in the Sourcers folders are imported.
These modules contain classes for preprocessing, analysis, and visualization.
Paths are also defined where figures, tables, and processed data will be saved.

In [ ]:
import time
import psutil
import os
import pandas as pd

# Import modules
from Sourcers.Preprocessing import *
from Sourcers.Analysis import *
from Sourcers.Visualization import *

# Paths
IMAGES_PATH = os.path.join("Results", "Figures")
TABLES_PATH = os.path.join("Results", "Tables")
PROCESSED_DATA_PATH = os.path.join("Data", "Processed")
URL = "https://raw.githubusercontent.com/Yeikeer/IndoorCO2Map-Analysis/refs/heads/main/Data/Raw/indoorco2mapData.json"


### 2. Data loading
The DataLoad class is used to download a JSON file from a web link and automatically convert it into a DataFrame.
This method allows you to centralize data loading and always work with a uniform format.

In [ ]:
# Load Data-------
df = DataLoad(URL)()

### 3. Data cleaning

The DataCleaner class:
Replaces empty strings, “None,” “NaN,” etc.

Imputes missing values:
- Mean/median for numeric variables
- Mode for categorical variables
- Ignores columns containing lists or dictionaries.

This function leaves the dataset ready for further analysis.

In [ ]:
# Cleaning-------
df = DataCleaner(df)()

### 4. Construction of time series

The TimeSeries class reconstructs the time series for each measurement:

- It uses the start time (startOfMeasurement).
- It applies the measurement interval (interval).
- It generates the time_list with absolute or relative timestamps.
  
This is necessary for temporal analysis and derived calculations such as slopes or curvatures.

In [ ]:
# Time Series------- 
ts = TimeSeries(df)
df = ts("co2readings", "startOfMeasurement", "interval", absolute=True)

### 5. Feature extraction

Relevant statistics are generated from the CO₂ and time lists.

5.1 Basic statistical features

Automatically generates columns such as:

- Median (Med)
- Standard deviation (Std)
- Variance (Var)
- Minimum and maximum
- Range
- RMS
- Coefficient of variation (CV)

These features allow measurements to be compared between devices or locations.

5.2 Temporal characteristics

Calculates dynamic indicators:

- Slope (CO₂ trend)
- Curvature (acceleration or shape of the curve)
- Magnitude of differences between readings (temporal variability)

These metrics allow us to understand the dynamics of CO₂ over time.

In [ ]:
# Features--------
df = FeatureExtractor(df, list_col="co2readings")()
df = TemporalFeatureExtractor(df, co2_col="co2readings", time_col="timelist")()

### 6. Classification

Here, categorical labels are created to facilitate segmentation of the analysis.

6.1 Classification by CO₂ level
The ClassifierCO2 class divides measurements into ranges:

- Safe
- Moderate
- High
- Risky
- Dangerous
- Severe
- Critical
- Lethal

Classes with fewer than 10 records are then removed to avoid bias in the analysis.

6.2 Ventilation Classification

Defines whether the site has:

- Natural ventilation
- Mechanical ventilation
- Both
- None

6.3 Time of Day Classification

Groups each measurement into:

- Midnight
- Morning
- Noon
- Afternoon

This integrates time context into the analysis.

In [ ]:

# Classification--------
# -CO2 Level Classification 
df["co2class"] = df["co2readingsMed"].apply(ClassifierCO2.classify_co2)
counts = df["co2class"].value_counts()
valid_classes = counts[counts >= 10].index.tolist()
df = df[df["co2class"].isin(valid_classes)].copy()
# -Ventilation Classification 
df["ventilationclass"] = df.apply(VentilationClassifier.classify_ventilation, axis=1)
# -Time of Day Classification
tod = TimeOfDayClassifier()
df["timeday"] = df["timelist"].apply(tod.classify_list)

### 7. Count by country + Bar chart

PlotBuilder allows you to generate charts in a unified way.
Here we use the bar method, which follows a general structure:

X-axis: categories (countries)

Y-axis: counts

Basic customization (title, orientation)

Other types of charts can also be easily created in the same format:

- pb.hist() → histograms
- pb.box() → boxplots
- pb.line() → lines
- pb.heatmap() → heat maps
- pb.scatter() → scatter plot
- pb.violin() → violin plots

In [ ]:
#-count by country
country_counts = df["countryName"].value_counts().reset_index()
country_counts.columns = ["countryName", "count"]
country_counts10 = country_counts[country_counts["count"] >= 10].reset_index(drop=True)

# Bar plot of counts by country
pb = PlotBuilder(country_counts10)
fig = pb.bar(
    x="countryName",
    y="count",
    title="Count by country",
    horizontal=True
)
FigureExporter(fig).save(os.path.join(IMAGES_PATH, "count_country.png"))

![Count by country](Results/Figures/count_country.png)

### 8. CO₂ distribution by class

Violin plots show the complete distribution of a numerical value according to a categorical group.

They use smoothed densities to visualize:

- Variability
- Concentration
- Symmetries or long tails

It is ideal for comparing the dispersion between CO₂ classes.

In [ ]:
#-CO2 levels by CO2 class
#Violin plot of CO2 levels by CO2 class
pb4 = PlotBuilder(df)

fig4 = pb4.violin(
    column="co2readingsAvg",   # variable numérica
    by="co2class",             # variable categórica
    title="CO2 Distribution by CO2 Class"
)

FigureExporter(fig4).save(os.path.join(IMAGES_PATH, "violin_co2class.png"))

![CO2 Distribution by CO2 Class](Results/Figures/violin_co2class.png)

### 9. Statistical tests

Nonparametric tests are used here to evaluate differences between groups.

9.1 Kruskal–Wallis test

This is used when:

- Multiple groups are compared
- The data are not normal
- You want to know if there are statistically significant differences
- It is applied to each important numerical variable.

The result returns:

- Statistic
- p-value
- Name of the test

and is then exported as a PNG table.

In [ ]:
# Numeric variables of interest
numeric_features = [
    "co2readingsMed",
    "co2readingsStd",
    "co2readingsCV",
    "co2readingsSlope",
    "co2readingsCurvature",
    "co2readingsStd_diff"
]

#-Kruskal-Wallis Test for differences between groups
kw_results = []
for feature in numeric_features:
    groups = df.groupby("co2class")[feature].apply(list).to_dict()
    test = KruskalWallisTest(groups)()
    test["feature"] = feature
    kw_results.append(test)
kw_df = pd.DataFrame(kw_results)[["feature", "test", "statistic", "pvalue"]]
TableExporter(kw_df).to_png(os.path.join(TABLES_PATH, "kruskal_tests.png"))

![Kruskal–Wallis test](Results/Tables/kruskal_tests.png)

### 10. Exporting processed data

The TableExporter class allows you to easily export:

- DataFrames or dictionaries to CSV
- Tables as PNG images

The final, fully processed dataset is saved here for later use.

In [ ]:
# Export processed data -------
TableExporter(df).to_csv(PROCESSED_DATA_PATH, "indoorco2map_processed.csv")

_____________________________________________________________________________________________________________________

## Conclusions

The notebook demonstrates how the classes implemented in the `Sourcers` module allow you to build a robust, reusable, and easy-to-maintain analysis pipeline. Each stage of the workflow is encapsulated in specialized objects, which facilitates its application in new projects or similar datasets.

The use of `DataLoad`, `DataCleaner`, and `TimeSeries` ensures a consistent foundation: from downloading to the temporal organization of readings. Subsequently, feature extractors enrich the data with statistical and dynamic metrics essential for CO₂ behavior analysis.

Classification classes add a semantic layer that makes it possible to segment data according to risk levels, ventilation type, or time slot. This enables comparative visualizations and statistical analyses such as the Kruskal–Wallis test, applied here to evaluate differences between categories.

Finally, the visualization and export modules close the flow by allowing the generation of figures and tables ready for publication or reporting.

Together, these functions provide a modular and extensible environment for the study of environmental data, enabling everything from basic inspection tasks to more advanced analysis with statistical methods and custom graphics.
